<a href="https://colab.research.google.com/github/annabashlakova/ECON3916-Statistical-Machine-Learning/blob/main/%5BLab_6%5D_The_Architecture_of_Bias.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# Step 1L Ingestion and Manual Shuffling
import seaborn as sns
import pandas as pd
import numpy as np

# 1. Data Ingestion
df = sns.load_dataset('titanic')
print(f"Total Population: {len(df)}")
print(f"Population Survival Rate: {df['survived'].mean():.4f}")

#2. Manual Shuffle (Simulation of Sampling)
np.random.seed(2026)
indices=np.random.permutation(df.index)


Total Population: 891
Population Survival Rate: 0.3838


In [6]:
split_point = int(len(df) * 0.8)
# Slicing the shuffled indices
train_idx = indices[:split_point]
test_idx = indices[split_point:]

# Creating Subsets
train_set = df.loc[train_idx]
test_set = df.loc[test_idx]

# Bias Check (The delta)
train_surv = train_set['survived'].mean()
test_surv = test_set['survived'].mean()
delta = abs(train_surv - test_surv)

print(f"Train Survival Rate: {train_surv:.4f}")
print(f"Test Survival Rate: {test_surv:.4f}")
print(f"Sampling Bias (Delta): {delta:.4f}")

Train Survival Rate: 0.3736
Test Survival Rate: 0.4246
Sampling Bias (Delta): 0.0510


In [8]:
# Fixing Covariate Shift
from sklearn.model_selection import train_test_split

# Stratify by 'pclass'
X_train, X_test = train_test_split(df,test_size=0.2,stratify=df["pclass"], random_state=42)

print("\n--- Stratified Split ---")
print("Train Class Dist:\n", X_train['pclass'].value_counts(normalize=True))
print("Test Class Dist:\n", X_test['pclass'].value_counts(normalize=True))


--- Stratified Split ---
Train Class Dist:
 pclass
3    0.550562
1    0.242978
2    0.206461
Name: proportion, dtype: float64
Test Class Dist:
 pclass
3    0.553073
1    0.240223
2    0.206704
Name: proportion, dtype: float64


In [9]:
import numpy as np
from scipy.stats import chisquare

# 1) Observed and expected arrays
observed = np.array([450, 550])   # [Control, Treatment]
expected = np.array([500, 500])   # planned 50/50 split over 1000 users

# 2) Chi-Square test (goodness-of-fit) + p-value
chi2_stat, p_value = chisquare(f_obs=observed, f_exp=expected)

# 3) Print results + conclusion
print(f"Observed: {observed.tolist()} | Expected: {expected.tolist()}")
print(f"Chi-square statistic: {chi2_stat:.4f}")
print(f"p-value: {p_value:.6f}")

if p_value < 0.01:
    print("CRITICAL FAILURE: Sample Ratio Mismatch (SRM) Detected. Check Load Balancer.")
else:
    print("Variance is within natural limits.")


Observed: [450, 550] | Expected: [500, 500]
Chi-square statistic: 10.0000
p-value: 0.001565
CRITICAL FAILURE: Sample Ratio Mismatch (SRM) Detected. Check Load Balancer.
